In [1]:
import csv, json, logging, random, re, sys, time
from datetime import datetime, timezone
from pathlib import Path
from urllib import robotparser

In [2]:
import requests, urllib.robotparser as rp
from bs4 import BeautifulSoup

In [3]:
 BASE = "https://www.jumia.com.eg"

In [17]:
CATEGORIES = {
    "phones-tablets": "/phones-tablets/",
    "health-beauty":  "/health-beauty/",
    "baby-products":  "/baby-products/",
}
MAX_PAGES_PER_CATEGORY = 8      # ~40 products/page -> up to ~320 per category
MIN_PRODUCTS_TARGET = 500       # assignment requirement per run
MAX_REQUESTS_PER_RUN = 1000     # hard cap from Rules of Engagement
DELAY_RANGE = (2.0, 5.0)        # seconds between requests (rule: 2-5 s)
MAX_RETRIES = 4
TIMEOUT = 30
# EDIT THIS: course code + your real contact email
USER_AGENT = "DSC2107-Course-Project (your.name@students.ucu.ac.ug)"

OUT_RAW = Path("data/raw")
OUT_LOG = Path("logs")
OUT_RAW.mkdir(parents=True, exist_ok=True)
OUT_LOG.mkdir(parents=True, exist_ok=True)


In [18]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
    handlers=[logging.FileHandler(OUT_LOG / "scraper.log", encoding="utf-8"),
              logging.StreamHandler(sys.stdout)],
)
log = logging.getLogger("jumia_eg")

session = requests.Session()
session.headers.update({"User-Agent": USER_AGENT,
                        "Accept-Language": "en-US,en;q=0.9"})
request_count = 0

In [19]:
class BlockedError(Exception):
        """Raised when the site blocks us. We STOP (rule f) - never bypass."""

def load_robots(today: str) -> robotparser.RobotFileParser:
    r = session.get(f"{BASE}/robots.txt", timeout=TIMEOUT)
    (OUT_RAW / f"robots_{today}.txt").write_text(r.text, encoding="utf-8")
    rp = robotparser.RobotFileParser()
    rp.parse(r.text.splitlines())
    log.info("robots.txt fetched (HTTP %s) and saved", r.status_code)
    return rp


In [20]:
# FETCH 
def polite_sleep():
    time.sleep(random.uniform(*DELAY_RANGE))

In [21]:
def fetch(url: str):
    """GET with retry + exponential back-off. Returns HTML or None."""
    global request_count
    for attempt in range(1, MAX_RETRIES + 1):
        if request_count >= MAX_REQUESTS_PER_RUN:
            raise BlockedError("Request cap reached for this run")
        polite_sleep()
        request_count += 1
        try:
            r = session.get(url, timeout=TIMEOUT)
        except requests.RequestException as e:
            wait = 2 ** attempt * 3
            log.warning("Network error %s (attempt %d). Waiting %ds", e, attempt, wait)
            time.sleep(wait)
            continue

        if r.status_code == 200:
            if "captcha" in r.text[:5000].lower():
                raise BlockedError(f"CAPTCHA shown at {url}")
            return r.text
        if r.status_code in (403, 401):
            raise BlockedError(f"HTTP {r.status_code} at {url}")
        if r.status_code == 404:
            log.info("404 (no more pages): %s", url)
            return None
        if r.status_code in (429, 500, 502, 503, 504):
            wait = 2 ** attempt * 10
            log.warning("HTTP %s at %s. Backing off %ds", r.status_code, url, wait)
            time.sleep(wait)
            continue
        log.warning("Unexpected HTTP %s at %s", r.status_code, url)
        return None
    log.error("Gave up on %s after %d attempts", url, MAX_RETRIES)
    return None

In [22]:
# PARSE 
def txt(el):
    return el.get_text(" ", strip=True) if el else None


In [23]:
def parse_card(card, category_key, page, position, scraped_at):
    a = card.select_one("a.core") or card.find("a", href=True)
    if a is None:
        return None
    href = a.get("href", "")
    rev_text = txt(card.select_one("div.rev"))
    reviews_raw = None
    if rev_text:
        m = re.search(r"\((\d[\d,]*)\)", rev_text)
        reviews_raw = m.group(1) if m else None
    return {
        "scraped_at": scraped_at,
        "site": "jumia.com.eg",
        "category_scraped": category_key,
        "page": page,
        "position": position,
        "product_id": a.get("data-gtm-id") or card.get("data-sku"),
        "name": txt(card.select_one("h3.name")) or a.get("data-gtm-name"),
        "brand": a.get("data-gtm-brand"),
        "category_path": a.get("data-gtm-category"),
        "price_raw": txt(card.select_one("div.prc")),
        "old_price_raw": txt(card.select_one("div.old")),
        "discount_raw": txt(card.select_one("div.bdg._dsct")),
        "rating_raw": txt(card.select_one("div.stars._s")),   # "4.5 out of 5"
        "review_count_raw": reviews_raw,
        "url": BASE + href if href.startswith("/") else href,
    }

In [24]:

def parse_listing(html, category_key, page, scraped_at):
    soup = BeautifulSoup(html, "html.parser")
    cards = soup.select("article.prd")
    out = []
    for i, c in enumerate(cards, 1):
        try:
            rec = parse_card(c, category_key, page, i, scraped_at)
            if rec and rec["name"]:
                out.append(rec)
        except Exception as e:           # one bad card must not crash the run
            log.warning("Card parse failed (%s p%d #%d): %s", category_key, page, i, e)
    return out

In [25]:
# MAIN
def run():
    now = datetime.now(timezone.utc)
    today = now.strftime("%Y-%m-%d")
    scraped_at = now.isoformat(timespec="seconds")
    log.info("=== Run started %s ===", scraped_at)

    rp = load_robots(today)
    records, seen = [], set()

    try:
        for cat_key, path in CATEGORIES.items():
            cat_count = 0
            for page in range(1, MAX_PAGES_PER_CATEGORY + 1):
                url = f"{BASE}{path}" + (f"?page={page}#catalog-listing" if page > 1 else "")
                if not rp.can_fetch(USER_AGENT, url):
                    log.warning("robots.txt disallows %s - skipping", url)
                    break
                html = fetch(url)
                if html is None:
                    break
                items = parse_listing(html, cat_key, page, scraped_at)
                if not items:
                    log.info("No products on %s page %d - end of category", cat_key, page)
                    break
                for it in items:
                    key = it["url"]
                    if key in seen:
                        continue
                    seen.add(key)
                    records.append(it)
                    cat_count += 1
                log.info("%s page %d: %d cards (category total %d)",
                         cat_key, page, len(items), cat_count)
    except BlockedError as e:
        log.error("STOPPED: %s. Document this in your report and lower your rate.", e)
    except KeyboardInterrupt:
        log.error("Interrupted by user")

    # Always save whatever we collected (partial data is still evidence)
    json_path = OUT_RAW / f"jumia_eg_{today}.json"
    csv_path = OUT_RAW / f"jumia_eg_{today}.csv"
    if json_path.exists():                              # never overwrite raw evidence
        suffix = now.strftime("%H%M%S")
        json_path = OUT_RAW / f"jumia_eg_{today}_{suffix}.json"
        csv_path = OUT_RAW / f"jumia_eg_{today}_{suffix}.csv"
    json_path.write_text(json.dumps(records, ensure_ascii=False, indent=2), encoding="utf-8")
    if records:
        with open(csv_path, "w", newline="", encoding="utf-8-sig") as f:
            w = csv.DictWriter(f, fieldnames=list(records[0].keys()))
            w.writeheader()
            w.writerows(records)

    log.info("Saved %d records, %d requests used -> %s", len(records), request_count, json_path)
    if len(records) < MIN_PRODUCTS_TARGET:
        log.warning("Below target of %d products. Raise MAX_PAGES_PER_CATEGORY "
                    "or add a category (stay within request cap).", MIN_PRODUCTS_TARGET)
    log.info("=== Run finished ===")


if __name__ == "__main__":
    run()


2026-10-12 20:35:05,888 INFO === Run started 2026-10-12T17:35:05+00:00 ===
2026-10-12 20:35:06,328 INFO robots.txt fetched (HTTP 200) and saved
2026-10-12 20:35:20,953 INFO phones-tablets page 1: 108 cards (category total 102)
2026-10-12 20:35:33,732 INFO phones-tablets page 2: 40 cards (category total 139)
2026-10-12 20:36:06,726 WARNING Network error HTTPSConnectionPool(host='www.jumia.com.eg', port=443): Read timed out. (read timeout=30) (attempt 1). Waiting 6s
2026-10-12 20:36:21,427 INFO phones-tablets page 3: 40 cards (category total 176)
2026-10-12 20:36:33,478 INFO phones-tablets page 4: 40 cards (category total 211)
2026-10-12 20:36:40,546 INFO phones-tablets page 5: 40 cards (category total 246)
2026-10-12 20:36:46,604 INFO phones-tablets page 6: 40 cards (category total 286)
2026-10-12 20:36:51,666 INFO phones-tablets page 7: 40 cards (category total 325)
2026-10-12 20:36:59,227 INFO phones-tablets page 8: 40 cards (category total 365)
2026-10-12 20:37:16,820 INFO health-bea